# 응용 03 · 라벨 면적과 가로세로비 분석

`03_audit.py`의 입력과 계산 과정을 입문자 관점에서 설명합니다. **Shift+Enter**로 위에서부터 실행하세요. 이 문서는 CSV를 새로 저장하지 않고 데이터와 계산 결과를 화면에 표시합니다. 검증 여부와 판단은 마지막 기록란에 직접 작성하세요.

학습 순서: manifest 읽기 → 라벨 해석 → 면적·가로세로비 → 분할별 객체 수 → 그룹·해시 집계. 필요한 패키지는 numpy와 pandas입니다.

In [1]:
from pathlib import Path
import pandas as pd

roots = [Path.cwd(), *Path.cwd().parents]
BASE = next((root / "20261106/02_advanced" for root in roots
             if (root / "20261106/02_advanced/03_audit.py").is_file()), None)
if BASE is None:
    raise FileNotFoundError("저장소 내부에서 문서를 실행하세요.")
manifest = pd.read_csv(BASE / "data/manifest.csv")
print(manifest.head(3).to_string(index=False))


          image_id split            group_id                               image                               label  width  height  objects                                                           sha256
standard_train_000 train standard_train_g000 images/train/standard_train_000.png labels/train/standard_train_000.txt    160     128        3 42b07011d52176915426868403ad60c25cd337ee957a1d5eed4d387ced6deadb
standard_train_001 train standard_train_g000 images/train/standard_train_001.png labels/train/standard_train_001.txt    160     128        3 f1d81ed97338b9fdd2cf26e1b681258f5715e2a06de5a8583a82d689b09a73b0
standard_train_002 train standard_train_g001 images/train/standard_train_002.png labels/train/standard_train_002.txt    160     128        3 d6a33e9c532c8f00ba9eea3db1e9d22452e8b4191f5b2c8522bb43d1515d9cfc


## 1. manifest는 어떤 표인가요?

manifest는 이미지와 라벨 파일의 위치, 크기, 분할 정보를 연결한 목록입니다. DataFrame은 pandas의 표 자료형입니다. `pd.read_csv()`로 읽고 `.head(3)`으로 앞의 세 행을 볼 수 있습니다.

|열|의미|
|---|---|
|image_id|이미지 식별자|
|image / label|data 폴더에서 시작하는 파일 경로|
|width / height|원본 이미지의 가로·세로 픽셀 수|
|split|train, val, test 중 용도|
|group_id|같이 묶어 다루는 촬영·장면 등의 그룹|
|sha256|파일 내용에서 계산한 해시|

`itertuples()`는 표를 한 행씩 꺼내며 `image.width`처럼 열 이름으로 값을 읽게 합니다. `Path / 문자열`은 경로를 연결합니다.

In [2]:
image = next(manifest.itertuples())
label_path = BASE / "data" / image.label
print("라벨 경로:", label_path)
print("원본 크기:", image.width, image.height)
print("라벨 내용:\n" + label_path.read_text(encoding="utf-8"))


라벨 경로: /home/swchoi/projects/python/practice_assignment/20261106/02_advanced/data/labels/train/standard_train_000.txt
원본 크기: 160 128
라벨 내용:
0 0.50937500 0.64453125 0.16875000 0.21093750
1 0.19687500 0.35546875 0.16875000 0.21093750
2 0.75625000 0.27343750 0.18750000 0.23437500



## 2. 라벨 TXT 한 줄 읽기

YOLO 라벨은 한 객체를 `class_id cx cy w h` 한 줄로 표현합니다. cx·cy는 중심, w·h는 폭·높이이며 이미지 크기로 나눈 정규화 값입니다. class_id는 종류 번호입니다.

`read_text()`는 파일 내용을 문자열로 읽고, `splitlines()`는 줄 목록으로, `line.split()`은 공백으로 분리한 항목 목록으로 바꿉니다. `map(float, ...)`는 각 문자열을 소수로 바꿉니다. 클래스 번호도 이때 소수가 되므로 표에 넣을 때 `int(cls)`로 정수로 바꿉니다.

내용이 빈 라벨 파일은 객체가 없는 negative 이미지의 표현일 수 있습니다. 존재하지 않는 라벨 파일과 구분해서 기록합니다. 아래 반복문은 빈 파일에서 객체 행을 추가하지 않습니다.

In [3]:
line = label_path.read_text(encoding="utf-8").splitlines()[0]
cls, cx, cy, w, h = map(float, line.split())
print("클래스:", int(cls))
print("중심:", cx, cy, "정규화 폭·높이:", w, h)


클래스: 0
중심: 0.509375 0.64453125 정규화 폭·높이: 0.16875 0.2109375


## 3. 면적과 가로세로비 계산

**정규화 면적 = w × h**입니다. 원본 면적 `(w×W)×(h×H)`을 이미지 전체 면적 `W×H`로 나눈 값과 같습니다. 원본 구현에서는 이 값을 `area_ratio` 열에 저장합니다.

**원본 가로세로비 = (w × W) / (h × H)**입니다. W는 이미지 가로, H는 세로입니다. 이미지가 정사각형이 아니면 `w/h`와 픽셀 기준 가로세로비는 다릅니다.

예를 들어 이미지가 160×128이고 w=0.25, h=0.3125이면 박스는 40×40픽셀입니다. 정규화 폭·높이는 다르지만 원본 가로세로비는 1입니다. 양수 높이를 가진 라벨을 전제로 하는 원본 계산식을 그대로 따라갑니다.

In [4]:
example_w, example_h = 0.25, 0.3125
print("예시 정규화 면적:", example_w * example_h)
print("예시 원본 크기:", example_w * 160, example_h * 128)
print("예시 원본 가로세로비:", (example_w * 160) / (example_h * 128))


예시 정규화 면적: 0.078125
예시 원본 크기: 40.0 40.0
예시 원본 가로세로비: 1.0


## 4. A03 구현의 전체 흐름

`rows=[]`는 객체별 계산을 담을 빈 리스트입니다. `dict(...)`는 열 이름과 값을 짝지은 딕셔너리를 만들고 `append()`로 리스트에 한 행씩 넣습니다. 마지막에 `pd.DataFrame(rows)`로 표를 만듭니다.

바깥 반복문은 이미지마다, 안쪽 반복문은 이미지 속 객체마다 실행됩니다. 따라서 결과 표의 행 수는 이미지 수가 아니라 객체 수입니다.

In [5]:
rows = []
for image in manifest.itertuples():
    for line in (BASE / "data" / image.label).read_text(encoding="utf-8").splitlines():
        cls, cx, cy, w, h = map(float, line.split())
        rows.append(dict(image_id=image.image_id, split=image.split,
                         class_id=int(cls), area_ratio=w * h,
                         aspect_ratio=(w * image.width) / (h * image.height)))
r = pd.DataFrame(rows)
print("이미지 행 수:", len(manifest), "객체 행 수:", len(r))
print(r.head(6).round(6).to_string(index=False))


이미지 행 수: 60 객체 행 수: 150
          image_id split  class_id  area_ratio  aspect_ratio
standard_train_000 train         0    0.035596           1.0
standard_train_000 train         1    0.035596           1.0
standard_train_000 train         2    0.043945           1.0
standard_train_001 train         0    0.053174           1.0
standard_train_001 train         1    0.033008           1.0
standard_train_001 train         2    0.033008           1.0


## 5. 분할·클래스별 객체 수

`groupby(["split", "class_id"])`는 두 열의 값이 같은 행을 묶습니다. `.size()`는 묶음의 행 수를 세고 `unstack(fill_value=0)`은 클래스 번호를 가로 열로 옮깁니다. 비어 있는 조합은 0으로 표시합니다.

면적·비율도 클래스별로 묶어서 최소·평균·최댓값을 읽을 수 있습니다. 아래 집계는 이 입력 파일에서 계산한 수치를 보여줍니다.

In [6]:
print("분할·클래스별 객체 수:")
print(r.groupby(["split", "class_id"]).size().unstack(fill_value=0).to_string())
print("\n클래스별 면적·가로세로비:")
print(r.groupby("class_id")[["area_ratio", "aspect_ratio"]].agg(["min", "mean", "max"]).round(6).to_string())


분할·클래스별 객체 수:
class_id   0   1   2
split               
test      10  10  10
train     30  30  30
val       10  10  10

클래스별 면적·가로세로비:
         area_ratio                     aspect_ratio          
                min      mean       max          min mean  max
class_id                                                      
0          0.028125  0.045676  0.070508          1.0  1.0  1.0
1          0.030518  0.048651  0.070508          1.0  1.0  1.0
2          0.028125  0.046758  0.066846          1.0  1.0  1.0


## 6. 그룹과 파일 중복 집계

`manifest.groupby("group_id").split.nunique()`는 그룹마다 등장한 split 종류 수를 셉니다. 1보다 큰 그룹은 여러 분할에 등장한 그룹입니다. 조건식 결과의 True를 합하면 해당 그룹 수가 됩니다.

`sha256.duplicated()`는 앞에 같은 해시가 있으면 True입니다. 합계는 같은 해시가 반복되는 추가 행 수이며 중복 그룹 수와 다릅니다. 파일 내용이 다른 비슷한 장면은 이 해시 집계만으로 알 수 없습니다.

In [7]:
split_counts = manifest.groupby("group_id")["split"].nunique()
print("여러 split에 등장한 그룹 수:", int((split_counts > 1).sum()))
print("동일 sha256의 추가 행 수:", int(manifest["sha256"].duplicated().sum()))


여러 split에 등장한 그룹 수: 0
동일 sha256의 추가 행 수: 0


## 7. 원본 실행과 직접 기록할 내용

원본 `03_audit.py`는 같은 계산 뒤 `outputs/label_audit.csv`를 저장합니다. `to_csv(..., index=False)`의 index=False는 DataFrame의 행 번호를 별도 열로 저장하지 않는다는 뜻입니다. 원본 파일을 실행하려면 저장소 루트에서 `.venv/bin/python 20261106/02_advanced/03_audit.py`를 사용합니다.

|기록 항목|직접 작성|
|---|---|
|실행 환경·명령||
|면적·비율 관찰 수치||
|그룹·해시 집계 수치||
|검증 여부·판단 근거||
|관찰과 구분할 원인 가설||

연습: 한 행의 정규화 폭·높이에 원본 가로·세로 길이를 곱해 박스 크기를 구해 보세요. 그다음 이미지 행 수와 객체 행 수가 다른 이유를 자신의 말로 정리해 보세요.